# 01 — Fit a sphere (Student workbook)

**Time:** 65 minutes
**Question:** what can a SANS curve tell us about a dilute spherical particle?

You will connect the analytical sphere amplitude to a SasModels fit, choose
bounds, and diagnose the result using normalized residuals.

> **Lecture connection**
>
> - Slide 18, *The SANS master equation*: the model you fit is this equation
>   with $S(Q)=1$ (dilute particles).
> - Slide 21, *The form factor, P(Q)*: the phase-map picture of why a sphere
>   gives minima at $QR \approx 4.49, 7.73, \ldots$ Rewatch the
>   [phase-map video](https://www.youtube.com/watch?v=4VKR-0b25Og) if the formula below feels abstract.
> - Slides 28 and 30, *Step 0: understand your sample* and *Fitting strategy*:
>   where defensible starting values and bounds come from.
> - Slide 31, *Is my fit good or not?*: what this notebook asks you to judge.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
values = np.genfromtxt(DATA / 'sphere_reference.csv', delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## Exercise 1 — Reduce the model

For a homogeneous sphere,

$$
F(q)=3\frac{\sin(qr)-qr\cos(qr)}{(qr)^3}, \qquad
I(q)=\mathrm{scale}\,V(\Delta\rho)^2F(q)^2+\mathrm{background}.
$$

Explain why volume is not an independent fifth shape parameter, then complete
the amplitude function. Ignore absolute scaling in this first visual test.

**How this maps onto the lecture.** SasModels writes the sphere as
$I(q) = \frac{\mathrm{scale}}{V}\,\left[3V\Delta\rho\,\frac{\sin(qr)-qr\cos(qr)}{(qr)^3}\right]^2
+ \mathrm{background}$, which is the expression above. Compare it with the
master equation on slide 18,
$I(Q) = \phi\,V_p\,(\Delta\rho)^2\,P(Q)\,S(Q) + \mathrm{bkg}$, where
$P(Q) = F(Q)^2$ is normalized so that $P(0)=1$. For dilute particles $S(Q)=1$,
and on an absolute scale `scale` *is* the volume fraction $\phi$. Because the
workshop data are in arbitrary units, `scale` here also absorbs a calibration
factor. That is one face of the $\phi \leftrightarrow \Delta\rho \leftrightarrow
\mathrm{scale}$ degeneracy from the lecture's part 2 wrap-up (slide 34). We fix
both SLDs so that `scale` carries the rest.

*Hint for the volume question:* write $V$ in terms of $r$.

In [ ]:
def sphere_amplitude(q, radius):
    qr = q * radius
    # TODO: implement 3 * (sin(qr) - qr*cos(qr)) / (qr**3)
    return np.ones_like(qr)


trial_radius = 50.0  # TODO: change this and rerun the plot
trial_shape = sphere_amplitude(q, trial_radius) ** 2


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(q, intensity / intensity.max(), 'o', ms=3, label='normalized data')
ax.loglog(q, trial_shape / trial_shape.max(), label=f'trial r = {trial_radius:g} Å')
ax.set(xlabel=r'$q$ / Å$^{-1}$', ylabel='normalized intensity')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


### Reading the size off the curve

Change `trial_radius` until the first minimum of your trial curve lines up
with the first minimum of the data. Then check yourself with the lecture's
rule of thumb (slide 21): the first zero of $F(qr)$ is at $qr \approx 4.493$,
so $r \approx 4.49/q_1$.

- The lecture's worked example used $R = 80$ Å, with minima at
  $Q = 0.0562, 0.0966, 0.136$ Å⁻¹. Are the data minima to the left or right of
  those? Is this sphere bigger or smaller than 80 Å?
- Why are the data minima not exactly zero? Think about noise, the background,
  and, for real samples, polydispersity and instrumental resolution (slide 21's
  dQ/Q panels and slide 39).

The radius you read off here is a *model-free* first estimate. Use it as the
starting value in the fit below rather than guessing.

## Exercise 2 — The fitting workflow

Every fit follows the same sequence:

1. create `SANSFitter()`;
2. load a reduced dataset with `load_data(path)`;
3. select a SasModels model with `set_model('model_name')`;
4. inspect its parameters with `get_params()`;
5. configure each parameter with `set_param(...)`;
6. call `fit(...)` and inspect the residuals.

The first two steps are demonstrated below. You will choose the model.

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(DATA / 'sphere_reference.csv'))


In [ ]:
# `set_model` takes a SasModels model name as a string.
# TODO: replace None with the model name for a homogeneous sphere.
model_name = None
if model_name is None:
    raise NotImplementedError('Choose a SasModels model')

fitter.set_model(model_name)

# `get_params` lists every parameter supplied by that model.
fitter.get_params()


### Changing parameters

The general form is:

```python
fitter.set_param(
    'parameter_name',
    value=initial_guess,
    min=lower_bound,
    max=upper_bound,
    vary=True,
)
```

- `value` is the optimizer's starting point, not the answer.
- `min` and `max` express physically defensible limits.
- `vary=True` lets the optimizer change the parameter.
- `vary=False` fixes independently known quantities such as SLD.

Keep both SLDs fixed. Choose and justify initial values and bounds for radius,
scale, and background. Afterward, call `get_params()` to audit your setup.

**Where do defensible numbers come from?** The lecture's *Step 0* (slide 28)
is to know what you put in the beam. In practice:

| Parameter | Lecture source for a starting value | Sensible bounds reflect… |
|---|---|---|
| `sld`, `sld_solvent` | Composition and density via an SLD calculator (slide 11) | fixed: known independently |
| `radius` | First minimum, $r \approx 4.49/q_1$ (slide 21), or Guinier, $R = \sqrt{5/3}\,R_g$ (slide 19) | synthesis, TEM, or DLS knowledge |
| `scale` | Height of the low-q plateau, $I(0) \propto \phi V \Delta\rho^2$ (slide 18) | concentration and calibration |
| `background` | Level of the flat high-q floor (slide 14) | must be ≥ 0 and below the data |

The SLD values used here are 4.2 for the particle and 1.0 for the solvent, in
units of 10⁻⁶ Å⁻², the "SLD units" of slide 11.

In [ ]:
# One fixed parameter is demonstrated below. `vary=False` tells the
# optimizer to retain this independently known value.
fitter.set_param('sld', value=4.2, vary=False)

# YOUR CODE HERE
# 1. Fix `sld_solvent` at 1.0.
# 2. Configure radius, scale, and background with an initial value and bounds.
# 3. Set `vary=True` for those three fitted parameters.
raise NotImplementedError('Configure the remaining model parameters')

fitter.get_params()


### Running the optimizer

`fit(engine='lmfit', method='least_squares')` performs a bounded local
least-squares fit. The returned dictionary contains the objective value and
fitted parameters. `plot_results(...)` overlays the model and displays
normalized residuals.

**What the optimizer actually does.** It searches for the parameters $\theta$
that minimize

$$
\chi^2(\theta) = \sum_i \left(\frac{I_i - I_{\mathrm{model}}(q_i;\theta)}{\mathrm{d}I_i}\right)^2 .
$$

Each term in brackets is a **normalized residual**: the misfit at one point,
measured in units of its own uncertainty. That is why `dI` matters (notebook
00). For a good model with correct error bars, about 68 % of normalized
residuals should lie between −1 and +1, few should exceed ±3, and they should
show no pattern along q ([SasView: *Assessing fit quality*](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/residuals_help.html)).

Two practical notes:

- The engine label `'lmfit'` is historical. Underneath, the helper calls
  [SciPy `least_squares`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html), a bounded trust-region method that only searches downhill from the
  starting point.
- With `engine='lmfit'`, `result['chisq']` is the **sum** above. The BUMPS
  engine reports the **reduced** value $\chi^2/(N-p)$, which is ≈ 1 for a good
  fit ($N$ data points, $p$ fitted parameters). Compare like with like.

In [ ]:
# `fit` accepts an optimization engine and an engine-specific method.
# For a bounded local fit, use engine='lmfit' and method='least_squares'.
# Store the returned dictionary in `result`, then call `plot_results`.

# YOUR CODE HERE
raise NotImplementedError('Fit the configured model and plot its residuals')


## Exercise 3 — Criticize, do not merely accept

Record:

- fitted radius and uncertainty;
- whether residuals look randomly distributed around zero;
- any fitted parameter touching a bound;
- one experimental systematic that this model ignores.

**Checklist for "is my fit good?" (slide 31).**

1. *Residuals:* scattered around zero with no runs or waves? A wave that
   follows the form-factor oscillations means the shape or size distribution
   is wrong, not the noise.
2. *Statistic:* is the reduced χ², $\chi^2/(N-p)$, close to 1? Much larger
   means a missing ingredient or underestimated `dI`. Much smaller means
   overestimated `dI` or overfitting.
3. *Bounds:* a parameter pinned at a bound has not been determined. The
   bound has decided it.
4. *Plausibility:* does the radius agree with your model-free estimate from
   the first minimum? Is the background below the lowest data points?

**Systematics a single monodisperse sphere ignores:** a size distribution
([SasView polydispersity](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/pd/polydispersity.html)) and instrumental smearing ([SasView resolution](https://www.sasview.org/docs/user/qtgui/Perspectives/Fitting/resolution.html)). Both fill in the minima (slide 21 shows 0, 5 and 10 % dQ/Q). The
lecture also covered reduction errors in transmission, empty-cell subtraction
and thickness (slides 42 and 44), and interparticle interactions, $S(Q) \neq 1$
(notebook 03). These synthetic data were generated without any of them. That is
why the minima are so sharp. Say which one you would expect to matter most for a
real measurement, and why.

In [ ]:
# YOUR CODE HERE
# Extract the fitted radius, scale, and background from `result`.
# Then write your scientific interpretation in a new Markdown cell.
raise NotImplementedError('Summarize and interpret the fit')


## Extension — Bad starting values

Change the initial radius to 15 Å while retaining broad bounds. Does the local
optimizer reach the same answer? If not, try BUMPS differential evolution with
`fitter.fit(engine='bumps', method='de', steps=200)`. What does this reveal
about optimization versus inference?

*Why this happens:* the χ² surface of a form factor is full of local minima.
A sphere of 15 Å puts its first minimum near $4.49/15 \approx 0.3$ Å⁻¹, so
a downhill search starting there can lock onto the wrong oscillation.
[Differential evolution](https://bumps.readthedocs.io/en/latest/guide/optimizer.html#differential-evolution) keeps a population of candidates spread over
the whole prior box, so it is far less sensitive to the start. Neither method
tells you *how well* the radius is determined. That question, the theme of slide
32 (*How can uncertainties help*), is taken up in notebook 04.

Remember the χ² note above when you compare the two engines' numbers.

## Further reading

- [Gommes, Jaksch & Frielinghaus, *Small-angle scattering for beginners*, J. Appl. Cryst. 54, 1832 (2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC8662971/), sections on the form factor and the sphere.
- [SasView `sphere` model documentation](https://www.sasview.org/docs/user/models/sphere.html): the exact
  equation, parameter units, and the meaning of `scale` on an absolute scale.
- J. S. Pedersen, *Analysis of small-angle scattering data from colloids and polymer
  solutions: modeling and least-squares fitting*, Adv. Colloid Interface Sci. 70, 171
  (1997): the standard reference for form factors, polydispersity and smearing in fits.